# The Vector Space Model and TF-IDF

This notebook is the *narrative* pillar: it imports the canonical reference implementation in `vector_space_model_tfidf.py` — which **owns every number** — and walks the topic section by section. Each mathematical claim is an `assert` in the `.py`; here we run those checks and print the numbers the page and the interactive viz mirror.

Run end to end with:

```bash
uv run --with numpy --with scikit-learn --with jupyter \
    jupyter execute notebooks/vector-space-model-tfidf/01_vector_space_model_tfidf.ipynb
```

In [ ]:
import sys, pathlib
_cands = [pathlib.Path.cwd(),
          pathlib.Path.cwd() / "notebooks" / "vector-space-model-tfidf",
          pathlib.Path("notebooks/vector-space-model-tfidf")]
for _d in _cands:
    if (_d / "vector_space_model_tfidf.py").exists():
        sys.path.insert(0, str(_d)); break
import vector_space_model_tfidf as vsm
index = vsm.build_inverted_index(vsm._FINANCE_CORPUS)
print("loaded reference implementation; query =", vsm._QUERY)

## 1. Documents as term-weight vectors

Each document becomes a sparse vector over the vocabulary. We reuse the BM25 finance corpus *verbatim* so the length-hijack flip below is provably the same phenomenon BM25 later fixes with its `b` parameter. Note the padded transcript's length.

In [ ]:
for doc, n in zip(index.doc_ids, index.doc_len.astype(int)):
    print(f'  {doc:<16} length={n:>3} tokens')

## 2. IDF is the self-information of a term (Theorem 1)

Draw a document uniformly at random; the surprise of finding term $t$ in it is $I(A_t) = -\log P(A_t) = \log(N/\mathrm{df}_t) = \mathrm{idf}_t$. A term in *every* document carries **0 bits** (here, `rate`); a singleton carries $\log N$. Scoring uses a smoothed form so universal terms keep a small weight — a convention, not the theorem.

In [ ]:
vsm.idf_table()
vsm.test_idf_is_self_information()

## 3. Sublinear term frequency (Proposition 1)

$g(\mathrm{tf}) = 1 + \log \mathrm{tf}$ is strictly increasing and strictly concave, so each extra occurrence of a term adds strictly less weight than the one before it.

In [ ]:
import numpy as np
for tf in (1, 2, 4, 8, 16):
    print(f'  tf={tf:>2}  1+log(tf)={1+np.log(tf):.3f}')
vsm.test_sublinear_tf_monotone_concave()

## 4. Cosine normalization removes pure magnitude (Proposition 2)

Scaling a document's counts by $c>0$ leaves its cosine to the query unchanged while the raw dot product scales by $c$ — cosine quotients out the magnitude the off-sphere divergence of *the retrieval problem* warned about.

In [ ]:
vsm.test_cosine_normalization_invariant()

## 5. The length-hijack flip

With the **raw tf-idf dot product**, the long padded transcript hijacks rank #1 by sheer length. Flip on **cosine normalization** and the concise on-point 10-K filing surfaces — the same winner BM25 reaches with `b = 0.75`.

In [ ]:
vsm.finance_demo()
vsm.test_length_hijack_flip()

## 6. Where TF-IDF falls short: the bridge to BM25 (Proposition 3)

Sublinear tf is **unbounded** — a single document repeating one term can still run away — whereas BM25's saturating factor is capped at $k_1+1$. That, plus a *tunable* length term, is what the next topic adds on the shared IDF factor introduced here.

In [ ]:
vsm.test_sublinear_unbounded_vs_bm25_bounded()

## 7. Cross-check against scikit-learn

The from-scratch cosine ranking agrees with `TfidfVectorizer` on the top document, so any drift would be a real bug rather than a formula difference.

In [ ]:
vsm.validate_against_sklearn()

---

**Three pillars, one set of numbers.** The rigorous math (the page), the interactive vector space laboratory, and this notebook all read from `vector_space_model_tfidf.py`. Change a number there and the viz and the page must change with it.